# Generate groudn truth dataset

In [2]:
# load autoreload
%load_ext autoreload
%autoreload 2


from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.
from openai import OpenAI
import json

# Add repo-local modules to sys.path
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path("04_evaluation/notebooks").resolve()

sys.path.append(str((NOTEBOOK_DIR / "../../01_agentic_rag").resolve()))
sys.path.append(str((NOTEBOOK_DIR / "../code").resolve()))
from ingestion import load_faq_data, build_index
from rag_helper import RAGBase
from evaluation_utils import llm_structured, calc_price, llm_structured_retry

### Load the FAQ data

In [62]:
# Load the FAQ data and build the index
documents = load_faq_data()

Loaded 6 courses
Fetching: https://datatalks.club/faq//json/data-engineering-zoomcamp.json
Added 404 documents from Data Engineering Zoomcamp (course: data-engineering-zoomcamp)
Fetching: https://datatalks.club/faq//json/stock-markets-analytics-zoomcamp.json
Added 93 documents from Stock Markets Analytics Zoomcamp (course: stock-markets-analytics-zoomcamp)
Fetching: https://datatalks.club/faq//json/ai-dev-tools-zoomcamp.json
Added 41 documents from AI Dev Tools Zoomcamp (course: ai-dev-tools-zoomcamp)
Fetching: https://datatalks.club/faq//json/llm-zoomcamp.json
Added 113 documents from LLM Zoomcamp (course: llm-zoomcamp)
Fetching: https://datatalks.club/faq//json/mlops-zoomcamp.json
Added 253 documents from MLOps Zoomcamp (course: mlops-zoomcamp)
Fetching: https://datatalks.club/faq//json/machine-learning-zoomcamp.json
Added 471 documents from ML Zoomcamp (course: machine-learning-zoomcamp)
Total documents loaded: 1375


In [63]:
# Filter for llm zoomcamp questions
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [64]:
documents = documents_llm

In [65]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


### Prepare the prompt for the LLM to generate questions based on the FAQ record

In [66]:
# struture the LLM outputs using pydantic models

class Questions(BaseModel):
    questions: list[str]

In [67]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long. 
Avoid em dashes and other special characters. 
Avoid using the words "question" or "answer" in the questions.
""".strip()

In [68]:
# Format user propmt as a json
user_prompt = json.dumps(doc)
user_prompt


'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [69]:
# Create a prompt message for the llm role developer data gen instructions and role user the json formatted faq record
messages = [{"role": "developer", "content": data_gen_instructions}, 
            {"role": "user", "content": user_prompt}]

In [70]:
# Open AI client
openai_client = OpenAI()

# Call models with the typed output
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [71]:
# Five questions generated by the LLM based on the FAQ record
result = response.output_parsed

print(result)

questions=['I just found this course, is it too late to join now?', 'Can I still take part even if I missed the start date?', 'If I join late, can I still get a certificate somehow?', 'Do I have to submit the project before submissions close to get the certificate?', 'Is it okay to start the course now, or is it already over?']


In [72]:
# Using an utility function to generate questions
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course, can I still start it now?', 'Is it too late to join the course if I discovered it recently?', 'Can I enroll after the course has already started?', 'If I join late, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I join now?']


In [73]:
usage.input_tokens, usage.output_tokens

(233, 82)

In [74]:
# Calculate the cost of the API call based on the usage
cost = calc_price(usage)
cost

{'input_cost': 0.00017475, 'output_cost': 0.000369, 'total_cost': 0.00054375}

In [75]:
# Create ground truth records
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course, can I still start it now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to join the course if I discovered it recently?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I join now?',
  'document': '74eb249bbf'}]

### Generate Ground Truth Records for all documents

In [76]:
# Function to generate json for each doc, call llm and create the recor
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [77]:
# test for first 5 documents
ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [78]:
ground_truth

[{'question': 'I just found this course, can I still join now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course if I discovered it after it began?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take part in the course even though I found it late?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to receive a certificate if I start the course late?',
  'document': '74eb249bbf'},
 {'question': 'I signed up for the LLM Zoomcamp, but when should I get the confirmation email?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need to wait for any registration confirmation before starting the course?',
  'document': '977bf7786c'},
 {'question': 'Can I begin watching lessons and submitting homework even if I never got an email after registering?',
  'document': '977bf7786c'},
 {'question': 'Is the registration tied to some 

#### Running parallel processing to generate ground truth for all documents

In [79]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# Create a thread pool with 6 workers and use map_progress to generate ground truth for all documents
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)


  0%|          | 0/113 [00:00<?, ?it/s]

In [80]:
results

[([{'question': 'Can I still join the course if I found it late?',
    'document': '74eb249bbf'},
   {'question': 'Is it too late to start this course now?',
    'document': '74eb249bbf'},
   {'question': 'If I join after the course has already started, can I still get a certificate?',
    'document': '74eb249bbf'},
   {'question': 'Do I need to finish and submit the project before submissions close to receive the certificate?',
    'document': '74eb249bbf'},
   {'question': 'Can latecomers take the course, and what do I need to do for the certificate?',
    'document': '74eb249bbf'}],
  ResponseUsage(input_tokens=233, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=324)),
 ([{'question': 'I signed up for the LLM Zoomcamp but never got a confirmation email, do I still need one to join?',
    'document': '977bf7786c'},
   {'question': 'Do I have to wait for some 

In [81]:
# Split response in records and usages
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth), len(usages)

(565, 113)

In [82]:
# Calculate the total cost of all API calls based on the usages
total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08618399999999997

In [83]:
# Utility function to calculate the total price of all API calls based on the usages
from evaluation_utils import calc_total_price
calc_total_price(usages)

0.08618399999999997

In [97]:
import pandas as pd

# create a dataframe
df_ground_truth = pd.DataFrame(ground_truth)

In [98]:
df_ground_truth

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Is it too late to start this course now?,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to finish and submit the project bef...,74eb249bbf
4,"Can latecomers take the course, and what do I ...",74eb249bbf
...,...,...
560,How do I get requests installed as v2.32.3 ins...,4b30b918bc
561,What pip command should I use to install reque...,4b30b918bc
562,"If lancedb needs a newer requests, how can I f...",4b30b918bc
563,Why am I still getting the wrong requests vers...,4b30b918bc


In [4]:
# Create a data folder if not existent and save file as csv
data_dir = (NOTEBOOK_DIR / "../data").resolve()
data_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Save the ground truth dataframe to a CSV file in the data directory
df_ground_truth.to_csv(data_dir / "ground_truth.csv", index=False)

In [5]:
# Load the ground truth data from the csv file
df_ground_truth = pd.read_csv(data_dir / "ground_truth.csv")

### Setup search

In [6]:
csv_path = (NOTEBOOK_DIR / "../data/ground_truth.csv").resolve()

if csv_path.exists():
    df_ground_truth = pd.read_csv(csv_path)
elif "ground_truth" in globals():
    # fallback if CSV was not saved yet in this session
    df_ground_truth = pd.DataFrame(ground_truth)
else:
    raise FileNotFoundError(f"ground_truth.csv not found at: {csv_path}")

In [7]:
# Convert the DataFrame to a list of dictionaries
ground_truth = df_ground_truth.to_dict(orient="records")

In [8]:
# Create a minsearch index for the llm-zoomcamp documents
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Loaded 6 courses
Fetching: https://datatalks.club/faq//json/data-engineering-zoomcamp.json
Added 404 documents from Data Engineering Zoomcamp (course: data-engineering-zoomcamp)
Fetching: https://datatalks.club/faq//json/stock-markets-analytics-zoomcamp.json
Added 93 documents from Stock Markets Analytics Zoomcamp (course: stock-markets-analytics-zoomcamp)
Fetching: https://datatalks.club/faq//json/ai-dev-tools-zoomcamp.json
Added 41 documents from AI Dev Tools Zoomcamp (course: ai-dev-tools-zoomcamp)
Fetching: https://datatalks.club/faq//json/llm-zoomcamp.json
Added 113 documents from LLM Zoomcamp (course: llm-zoomcamp)
Fetching: https://datatalks.club/faq//json/mlops-zoomcamp.json
Added 253 documents from MLOps Zoomcamp (course: mlops-zoomcamp)
Fetching: https://datatalks.club/faq//json/machine-learning-zoomcamp.json
Added 471 documents from ML Zoomcamp (course: machine-learning-zoomcamp)
Total documents loaded: 1375
Building index with 113 documents
Text fields: ['question', 'sectio

In [9]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

### Collect relevance data for each question in the ground truth records

In [10]:
# Take one ground truth record and run the search to get the top 5 results
q = ground_truth[0]
q

{'question': 'Can I still join the course if I found it late?',
 'document': '74eb249bbf'}

In [11]:
# Run the text search for the question in the ground truth record
doc_id = q["document"]
results = text_search(query=q["question"])

In [12]:
# Compare the results with the ground truth document id
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


In [13]:
# Collect relevance data for each question in the ground truth records
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance


[1, 0, 0, 0, 0]

In [14]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [15]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Can I still join the course if I found it late?


[1, 0, 0, 0, 0]

In [16]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I watch the live session for Office Hours, and do I need the Zoom link?


[1, 0, 0, 0, 0]

In [17]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I check the LLM Zoomcamp syllabus, homework, deadlines, and my progress?


[1, 0, 0, 0, 0]

In [18]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [19]:
# Compute the relevance for the first 15 ground truth records
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [20]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [21]:
# helper function to compute relevance for a single question and a given search function
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [22]:
# Compute the relevance for all ground truth records
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [23]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [24]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

### Search evaluation metrics

#### Hit rate

In [27]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [29]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [32]:
hit_rate = cnt / len(example)*100
hit_rate

93.33333333333333

In [37]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)*100

In [38]:
hit_rate(example)

93.33333333333333

In [ ]:
def hit_rate_numpy(relevance):
    relevance_array = np.array(relevance)
    return np.mean(relevance_array.max(axis=1)) * 100


In [40]:
hit_rate_numpy(example)

np.float64(93.33333333333333)

#### Mean reciprocal rank (MRR)

In [42]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [43]:
mrr(example)
# 0.822

0.8222222222222222

In [45]:
relevance_array = np.array(example)

In [49]:
# Create a 1 array the size of the relevance array
ones_array = np.ones_like(relevance_array)
# Divide the array by the index of columns + 1 to get the reciprocal rank
mrr_score = ones_array / (np.arange(relevance_array.shape[1]) + 1)
mrr_score


array([[1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.33333333, 0.25      , 0.2       ],
       [1.        , 0.5       , 0.

In [51]:
 mrr_array = mrr_score * relevance_array
 mrr_array

array([[1.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.5       , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.5       , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.33333333, 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.

In [ ]:
# mrr score sum of the total of the mrr_array divided by the number of questions in the relevance array
mrr_score = np.sum(mrr_array, axis=None) / relevance_array.shape[0]
mrr_score

np.float64(0.8222222222222222)

In [64]:
def mrr_numpy(relevance):
    relevance_array = np.array(relevance)
    reciprocal_ranks = 1 / (np.arange(relevance_array.shape[1]) + 1)
    mrr_array = relevance_array * reciprocal_ranks
    return np.sum(mrr_array) / relevance_array.shape[0]*100

In [65]:
mrr_numpy(example)

np.float64(82.22222222222221)

In [66]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate_numpy(relevance_total),
        "mrr": mrr_numpy(relevance_total),
    }

In [67]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': np.float64(84.42477876106194),
 'mrr': np.float64(71.73746312684366)}